# **⛑️ Safety Helmet Detection with YOLOv8**
In this lab, we will:
✅ **Use YOLOv26** for **safety helmet detection** in petrochemical environments  
✅ **Understand the dataset structure**  
✅ **Train a YOLOv26 model**  
✅ **Evaluate the model on the validation set**  
✅ **Run inference on test images**  

---

## **1️⃣ Understanding the Dataset Structure**
The dataset follows the **YOLO format**, which consists of:
📂 **train/** → Training images & labels  
📂 **valid/** → Validation images & labels  
📂 **test/** → Test images (for inference)  
📜 **data.yaml** → Defines dataset paths & class names  

We will use the **Construction-PPE** dataset from Ultralytics, which includes:
- Helmets, vests, gloves, boots, goggles
- **Missing equipment** classes (no_helmet, no_vest, etc.)

### https://docs.ultralytics.com/datasets/detect/construction-ppe

## **2️⃣ What's Inside a YOLO Label File?**
Each `.txt` file contains **annotations** in this format:

class_id | x_center | y_center | width height

✅ **All values are normalized** between **0 and 1**  
✅ The **bounding box** is defined by its **center** and **size**  

### **🔹 Example**
0 0.526 0.448 0.12 0.15
3 0.731 0.602 0.18 0.22

- **First column** → Class ID (`0` = boots, `3` = helmet, etc.)  
- **Rest** → Bounding box (normalized)

## **3️⃣ Loading the Dataset**

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset
import matplotlib.pyplot as plt

# Download the Construction-PPE dataset
dataset_info = check_det_dataset("construction-ppe.yaml")
dataset_path = "construction-ppe.yaml"

print("Classes:", dataset_info["names"])

In [ ]:
# Check dataset information
from pathlib import Path

ds_root = Path(dataset_info["path"])
train_images = list((ds_root / "images" / "train").glob("*"))
val_images = list((ds_root / "images" / "val").glob("*"))
test_images = list((ds_root / "images" / "test").glob("*"))

print(f"Train images: {len(train_images)}")
print(f"Val images:   {len(val_images)}")
print(f"Test images:  {len(test_images)}")

## **4️⃣ Training a YOLOv8 Model**
We will fine-tune a **pretrained YOLOv8 model**.

In [ ]:
# Load YOLOv26 model (small version)
model = YOLO("yolo26n.pt")

In [ ]:
# Run inference BEFORE training
results = model(str(test_images[1]), save=True)


predicted_image = results[0].plot()

plt.figure(figsize=(10, 10))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image (Before Training)")
plt.show()

#### The model doesn't thinks humans are horses... Let's try to train it! 🚀

In [ ]:
# Train on the Construction-PPE dataset
model.train(data=dataset_path, epochs=20, imgsz=640)

## **5️⃣ Evaluating the Model**
We use **mAP@0.5:0.95** to assess performance.

In [ ]:
# Run validation
metrics = model.val(data=dataset_path)

## **6️⃣ Running Inference on Test Images**

In [ ]:
# Load best weights and run inference
model = YOLO(model.trainer.best)
results = model(str(test_images[1]), save=True)

predicted_image = results[0].plot()

plt.figure(figsize=(10, 10))
plt.imshow(predicted_image)
plt.axis("off")
plt.title("Predicted Image (After Training)")
plt.show()

### 🚀 **Now you have a working YOLOv8 object detection pipeline for safety helmets!**

###Contributed by: Yazan Alshoibi